# Jais Arabic Voice Chatbot — مباشرة على كولاب 🇸🇦🎙️

بيئة تشغيل جاهزة للتشغيل المباشر على **T4 (16 GB)**:

`Browser (WebRTC) → Jais-ASR (STT) → Jais-3 (LLM) → Audar-TTS (TTS)`

**الخطوات:**
1. `Runtime → Change runtime type → GPU (T4)`
2. `Run all` (أول مرة ≈ 10 دقائق: pip + ~7 GB نماذج)
3. الخلية 6 تفتح لك طرفًا جديدًا — اضغط **Connect** وابدأ الكلام بالعربية 🎙️

⚠️ **خلية 3 إجبارية**: أحد النماذج (neuphonic/neucodec) مُقيّدة على HF وتحتاج تسجيل دخول + قبول ترخيص مرة واحدة.

[&#8593; افتح هذا الدفتر في Colab ](https://colab.research.google.com/github/911houssem/jais-voice-chatbot/blob/main/notebooks/jais_voice_chatbot_colab.ipynb)

In [ ]:
#@title 1) فحص كرت الشاشة واختيار الجودة المناسبة {display-mode:"form"}
import subprocess, re

def gpu_state():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader"]).decode()
        name, total, free = [x.strip() for x in out.split(",")]
        return name, int(re.sub(r"\D", "", total)) / 1024.0, int(re.sub(r"\D", "", free)) / 1024.0
    except Exception as e:
        print("No GPU detected:", e)
        return "", 0.0, 0.0

name, total_gb, free_gb = gpu_state()
print(f"GPU: {name} | total {total_gb:.1f} GB | free {free_gb:.1f} GB")
if free_gb and free_gb < 20:
    print("Less than 20 GB free -> JAIS_ASR_MODEL=auto picks the W4A16-AWQ ASR checkpoint.")
else:
    print("20+ GB free -> JAIS_ASR_MODEL=auto picks the fp16 Jais-ASR checkpoint.")

In [ ]:
#@title 2) استنساخ المشروع وتثبيت الاعتمادات (~5 دقائق) {display-mode:"form"}
REPO_URL = "https://github.com/911houssem/jais-voice-chatbot.git" #@param {type:"string"}

import os, subprocess
repo_dir = "/content/jais-voice-chatbot"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
os.chdir(repo_dir)
print("cwd:", os.getcwd())

# Runner extra pulls the prebuilt WebRTC UI; compressed-tensors is needed by
# the W4A16-AWQ ASR checkpoint auto-selected on 16 GB GPUs.
!pip install -q -r requirements.txt
print("Install done.")

## 3) رمز Hugging Face — **إجباري** 🔑
مستودع الـTTS codec [`neuphonic/neucodec`](https://huggingface.co/neuphonic/neucodec) مُقيّد (gated).

1. افتح <https://huggingface.co/neuphonic/neucodec> وسجّل دخولك واضغط زر **Agree and access repository** (مرة واحدة).
2. أنشئ رمز قراءة من <https://huggingface.co/settings/tokens> (اختَر **Read**).
3. الصقه في الخلية التالية وشغّلها.

In [ ]:
#@title 3) Hugging Face token (required for gated codec) {display-mode:"form"}
import os
TOKEN = "" #@param {type:"string"}
assert TOKEN.strip(), "الصق توكن HF من https://huggingface.co/settings/tokens أولًا"
os.environ["HF_TOKEN"] = TOKEN
from huggingface_hub import login
login(TOKEN, add_to_git_credential=False)

# gate check for the codec repo
try:
    import requests
    r = requests.get("https://huggingface.co/neuphonic/neucodec/resolve/main/.gitattributes",
                     headers={"Authorization": f"Bearer {TOKEN}"}, timeout=15)
    (r.raise_for_status())
    print("Token OK ✔ — gate accepted ✔")
except Exception as e:
    print("❌ الرمز لا يفتح المستودع المقيّد. افتح الرابط واضغط 'Agree and access repository',")
    print("ثم أنشئ توكن جديد وأعد تشغيل هذه الخلية. التفاصيل:", e)
    raise

In [ ]:
#@title 4) تحميل النماذج مسبقًا (~7 GB — يتسارع أول اتصال) {display-mode:"form"}
# Requires: cell 3 done (gated codec needs auth).
from huggingface_hub import snapshot_download

REPOS = ["IFM/Jais-ASR-W4A16-AWQ",        # STT quantized (T4 default)
         "inception42/Jais-3-0.9B",        # chat LLM
         "audarai/Audar-TTS-V1-Flash",     # TTS backbone + voices
         "neuphonic/neucodec"]             # TTS codec (gated -> needs token)

for repo in REPOS:
    print("downloading", repo, "...")
    try:
        snapshot_download(repo_id=repo)
        print("  ✔", repo)
    except Exception as e:
        print("  ❌ فشل تحميل", repo, "— تخطيه. الخطأ:", str(e)[:300])
print("Model cache step done (any ❌ above will retry automatically on first use).")
# On a 20GB+ GPU the app swaps to IFM/Jais-ASR (fp16) instead, which downloads
# lazily on first connection.

In [ ]:
#@title 5) تشغيل خادم الصوت (WebRTC عبر TURN relay) {display-mode:"form"}
APP_PORT = 8080 #@param {type:"integer"}

import os, subprocess
assert os.environ.get("HF_TOKEN"), "شغّل خلية 3 (توكن HF) قبل هذه الخلية"

# WebRTC media cannot enter the Colab VM over UDP, so both peers negotiate via
# these relay servers (free openrelay.metered.ca TURN over TCP). pipecat feeds
# PIPECAT_ICE_SERVERS to the bot AND to the browser through /start.
os.environ["JAIS_ASR_MODEL"] = "auto"          # AWQ checkpoint on T4 (~3.4 GB)
os.environ["JAIS_LLM_DTYPE"] = "fp16"
os.environ["JAIS_TTS_VOICE"] = "demo_female_1"
os.environ["PIPECAT_ICE_SERVERS"] = '[{"urls":["stun:stun.l.google.com:19302","turn:openrelay.metered.ca:80","turn:openrelay.metered.ca:443","turns:openrelay.metered.ca:443?transport=tcp"],"username":"openrelayproject","credential":"openrelayproject"}]'

logf = open("/content/jais-voice.log", "a")
proc = subprocess.Popen(
    ["python", "app.py", "-t", "webrtc", "--host", "0.0.0.0", "--port", str(APP_PORT)],
    stdout=logf, stderr=logf,
)
print("server started (PID %d)" % proc.pid)

In [ ]:
#@title 6) افتح واجهة الدردشة (نافذة جديدة) {display-mode:"form"}
import time, requests
from google.colab.output import eval_js, serve_kernel_port_as_window

for _ in range(90):
    try:
        if requests.get(f"http://127.0.0.1:{APP_PORT}/docs", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(2)
else:
    print("Uvicorn not up yet — check /content/jais-voice.log, then re-run this cell.")

proxy_base = eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
print("Browser URL:", proxy_base.rstrip('/') + "/client")
print("\nأو اضغط الزر أدناه لفتح طرف مباشر في /client:")
serve_kernel_port_as_window(port=APP_PORT)

## Troubleshooting
- **`GatedRepoError 401 neuphonic/neucodec`** → أنشئ استخدام خلية 3: قَبِل الترخيص من صفحة المستودع ثم أنشئ توكن. // قبِل الرابطة في صفحة النموذج ثم توكن.
- **OOM** → خلي `JAIS_ASR_MODEL=auto` (AWQ ~3.4 GB) وقلل `JAIS_LLM_MAX_NEW_TOKENS`.
- **Slow replies on T4** → متوقع مع ASR 6B مضغوط (3-8 ثوان/جملة).
- **`TTS emitted no speech tokens`** → عيّن `JAIS_TTS_REF_WAV` + `JAIS_TTS_REF_TEXT`.
- **Browser can't connect** → أول Connect يسخِّن النماذج (~1-2 دقيقة)؛ أعد المحاولة، واستخدم فقط رابط `https://…proxy.googlers.com` المطبوع.
- سجلات الخادم: `!tail -50 /content/jais-voice.log`